In [0]:
# Imports
import requests
import time
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import lit, current_timestamp

print("Imports successful")

In [0]:
# Definition of indicators
# We pull two population indicators:
# SP.POP.TOTL = total population (our main denominator)
# SP.URB.TOTL.IN.ZS = urban population % (urbanisation affects
#                     vulnerability differently than rural poverty)

COUNTRIES  = "KE;UG;TZ;ET;GH;NG;ZA;SN;ML;MZ;RW;ZM;ZW;CM;CI"
BASE_URL   = "https://api.worldbank.org/v2/country"

INDICATORS = [
    {
        "code": "SP.POP.TOTL",
        "name": "total_population"
    },
    {
        "code": "SP.URB.TOTL.IN.ZS",
        "name": "urban_population_pct"
    },
]

print(f"Targeting {len(INDICATORS)} population indicators")
print(f"Countries: {COUNTRIES}")

In [0]:
# Cell 3 — fetch and flatten functions
# Identical pattern to notebooks 01 and 03
# This is intentional — consistent patterns across notebooks
# makes the codebase readable and maintainable
# In Silver we'll unify all four sources

def fetch_world_bank(indicator_code: str, countries: str) -> list:
    """Fetches a World Bank indicator for all target countries."""
    url    = f"{BASE_URL}/{countries}/indicator/{indicator_code}"
    params = {
        "format":   "json",
        "per_page": 500,
        "date":     "2010:2023",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data[1] if data and len(data) > 1 else []


def flatten_records(records: list, indicator_name: str) -> list:
    """Flattens nested API response into row-level dicts. Skips nulls."""
    flat = []
    for r in records:
        if r.get("value") is None:
            continue
        flat.append({
            "country_code":   r["country"]["id"],
            "country_name":   r["country"]["value"],
            "country_iso3":   r.get("countryiso3code", ""),
            "indicator_id":   r["indicator"]["id"],
            "indicator_name": indicator_name,
            "year":           int(r["date"]),
            "value":          float(r["value"]),
        })
    return flat

# Test on total population
test = fetch_world_bank(INDICATORS[0]["code"], COUNTRIES)
flat_test = flatten_records(test, INDICATORS[0]["name"])
print(f"Test fetch: {len(flat_test)} records")
print(f"Sample: {flat_test[0]}")

In [0]:
# Fetch both indicators

all_flat = []

for ind in INDICATORS:
    print(f"Fetching: {ind['name']}...", end=" ")
    try:
        raw  = fetch_world_bank(ind["code"], COUNTRIES)
        flat = flatten_records(raw, ind["name"])
        all_flat.extend(flat)
        print(f"{len(flat)} records")
    except Exception as e:
        print(f"FAILED — {e}")
    time.sleep(2)

print(f"\nTotal records: {len(all_flat)}")

In [0]:
# Creating Spark DataFrame

schema = StructType([
    StructField("country_code",   StringType(),  True),
    StructField("country_name",   StringType(),  True),
    StructField("country_iso3",   StringType(),  True),
    StructField("indicator_id",   StringType(),  True),
    StructField("indicator_name", StringType(),  True),
    StructField("year",           IntegerType(), True),
    StructField("value",          DoubleType(),  True),
])

df_pop = spark.createDataFrame(all_flat, schema=schema)

df_pop = (df_pop
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source",      lit("world_bank_api"))
    .withColumn("domain",      lit("population"))
)

print(f"DataFrame: {df_pop.count()} rows, {len(df_pop.columns)} columns")
df_pop.printSchema()

In [0]:
display(df_pop)

In [0]:
# Write to Bronze Unity Catalog table

BRONZE_TABLE = "workspace.default.bronze_world_bank_population"

(df_pop.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Written to Bronze: {BRONZE_TABLE}")

In [0]:
# Verify

df_check = spark.read.table(BRONZE_TABLE)

print(f"Total rows: {df_check.count()}")

display(
    df_check.groupBy("indicator_name")
            .count()
            .orderBy("indicator_name")
)

In [0]:
# Sanity check on population values
# Total population for these countries ranges from
# ~14M (Rwanda) to ~220M (Nigeria)
# Urban % should be between 15–70% for Sub-Saharan Africa

from pyspark.sql.functions import round as spark_round, avg, min, max, format_number

print("Population value ranges:")
display(
    df_check.groupBy("indicator_name")
            .agg(
                spark_round(min("value"),  0).alias("min"),
                spark_round(avg("value"),  0).alias("avg"),
                spark_round(max("value"),  0).alias("max"),
            )
            .orderBy("indicator_name")
)

In [0]:
# Preview Kenya population trend
# Kenya's population should show steady growth 2010-2023

from pyspark.sql.functions import round as spark_round

display(
    df_check.filter(
        (df_check.country_code == "KE") &
        (df_check.indicator_name == "total_population")
    )
    .select("year", "country_name", "value")
    .orderBy("year")
)